# 목차 통합 & 후처리 파이프라인 (Phase 2)
**toc_docs(목차 데이터) → docs(1차 전처리 JSON) 병합 + 구조 정리 → final_data**

---

## 이 노트북에서 하는 일

기존 `preprocessing_pipeline_fixed.ipynb`가 만든 1차 결과물(`Preprocessed dataset/docs/D0XX.json`)에는
다음 4가지 문제가 있었습니다.

1. **목차(toc)가 제대로 안 뽑힘** — 로더가 자동 추출한 `toc`가 비어있거나 1~2개 항목뿐
2. **header_path가 진짜 목차와 안 맞음** — 본문 글머리표(□, 가/나) 기준으로 섹션이 나뉘어 있어 목차 번호(Ⅰ/1/2...)와 연결이 안 됨
3. **표의 병합 셀 중복 텍스트** — 셀이 합쳐졌던 표가 풀리면서 같은 텍스트가 여러 칸에 반복 추출됨
4. **비정상적으로 큰 섹션** — 한 섹션에 블록이 100개 넘게 몰려 있어 청킹 시 그대로 쓰기 어려움

이번에 AI 1차 추출 + 검수로 만든 **목차 데이터(`toc_docs/D0XX_toc.json`)**를 가지고,
위 4가지 문제를 **5단계(Phase 1~5)**로 나누어 순서대로 해결합니다.

## 작업 목차

| Phase | 작업 | 목적 | 효과 |
|---|---|---|---|
| Phase 0 | 드라이브 마운트 & 경로 설정 | 원본 99개 + 목차 99개를 동시에 읽을 수 있게 환경 구성 | 이후 모든 단계의 입출력 경로 확보 |
| Phase 1 | toc 병합 (doc_id 매칭) | 부실했던 `toc` 필드를 사람이 만든 진짜 목차로 교체 | 검색 시스템이 문서의 실제 목차 체계(Ⅰ/1/2...)를 알게 됨 |
| Phase 2 | header_path ↔ toc 매칭 | 본문 섹션과 목차 항목을 텍스트 유사도로 연결(`toc_ref`) | "2.1. 추진목표"처럼 목차 번호로 검색해도 본문 위치를 정확히 찾음 |
| Phase 3 | 표 병합 셀 중복 정리 | 같은 텍스트가 옆 칸에 반복된 표를 1회만 남기도록 정리 | 청크에 들어가는 불필요한 토큰 제거, 검색 노이즈 감소 |
| Phase 4 | 대형 섹션 분할 표시 | 블록 수가 많은 섹션에 `needs_subsplit` 플래그 부여 | 청킹 담당자가 추가 분할이 필요한 섹션을 바로 식별 가능 |
| Phase 5 | qa 재계산 & 최종 저장 | 병합·정리 후 통계(toc_header_match_rate 등) 재산출, `final_data`에 저장 | 100개 전체의 품질을 수치로 검증, 재작업 대상 자동 식별 |

## 경로 구조 (Google Drive)

```
내 드라이브
└─ part3data
   ├─ files                              # 원본 hwp/pdf 99건
   ├─ data_list.csv                      # 메타데이터
   └─ Preprocessed dataset
      ├─ docs                            # 1차 전처리 결과 (원본 파이프라인 산출물)  ← 입력 ①
      ├─ toc_docs                        # 사람이 만든 목차 JSON (D0XX_toc.json)     ← 입력 ②
      └─ final_data                      # 이 노트북의 최종 산출물 (이번에 새로 생성)  ← 출력
```

`docs`와 `toc_docs`는 **doc_id 키로 연결**합니다 (파일명이 아닌 JSON 내부의 `doc_id` 필드 기준 — 파일명 오기입 실수를 잡기 위한 이중 확인 포함).

## 제외 문서

`D026`, `D043`은 1차 파싱이 실패한 것으로 확인되어, 이번 병합 작업에서는 **건너뛰고** 별도 목록으로만 기록합니다.

---
##0.드라이브 마운트 & 경로 설정

기존 1차 파이프라인과 동일한 `BASE_DIR` 체계를 그대로 사용합니다. 새로 추가되는 경로는 `TOC_DIR`(목차 입력)과 `FINAL_DIR`(최종 출력)입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, re, unicodedata
from pathlib import Path
from difflib import SequenceMatcher

BASE_DIR   = Path('/content/drive/MyDrive/part3data')
OUTPUT_DIR = BASE_DIR / 'Preprocessed dataset'

DOCS_DIR  = OUTPUT_DIR / 'docs'        # 입력 ① : 1차 전처리 JSON
TOC_DIR   = OUTPUT_DIR / 'toc_docs'    # 입력 ② : 목차 JSON (D0XX_toc.json)
FINAL_DIR = OUTPUT_DIR / 'final_data'  # 출력    : 병합·정리 완료된 최종 JSON

FINAL_DIR.mkdir(parents=True, exist_ok=True)

# 1차 파싱 실패로 이번 작업에서 제외할 문서
EXCLUDE_DOC_IDS = {'D026', 'D043'}

print(f'docs   : {DOCS_DIR}')
print(f'toc_docs : {TOC_DIR}')
print(f'final_data : {FINAL_DIR}')
print(f'제외 문서 : {sorted(EXCLUDE_DOC_IDS)}')

In [ ]:
# ── 입력 파일 로드 ─────────────────────────────────────────────
doc_files = sorted(DOCS_DIR.glob('*.json'))
toc_files = sorted(TOC_DIR.glob('*_toc.json'))

print(f'docs 폴더 JSON 개수     : {len(doc_files)}')
print(f'toc_docs 폴더 JSON 개수 : {len(toc_files)}')

# doc_id 기준으로 두 딕셔너리 구성 (파일명이 아니라 내부 doc_id 키 사용)
docs_by_id = {}
for p in doc_files:
    with open(p, encoding='utf-8') as f:
        d = json.load(f)
    docs_by_id[d['doc_id']] = (p, d)

toc_by_id = {}
for p in toc_files:
    with open(p, encoding='utf-8') as f:
        t = json.load(f)
    toc_by_id[t['doc_id']] = (p, t)

print(f'docs_by_id     : {len(docs_by_id)}건')
print(f'toc_by_id      : {len(toc_by_id)}건')

# 양쪽에 모두 존재하는 doc_id만 처리 대상 (제외 문서는 여기서 미리 빠짐)
target_ids = sorted((set(docs_by_id) & set(toc_by_id)) - EXCLUDE_DOC_IDS)
missing_toc = sorted(set(docs_by_id) - set(toc_by_id) - EXCLUDE_DOC_IDS)
missing_doc = sorted(set(toc_by_id) - set(docs_by_id) - EXCLUDE_DOC_IDS)

print(f'\n처리 대상 : {len(target_ids)}건')
if missing_toc:
    print(f'⚠️ toc 누락 (docs에만 존재) : {missing_toc}')
if missing_doc:
    print(f'⚠️ doc 누락 (toc_docs에만 존재) : {missing_doc}')

---
##1.toc 병합 (doc_id 매칭)

### 왜 하는가
1차 산출물의 `toc` 필드는 로더가 자동 추출한 것이라 비어있거나 1~2개 항목뿐이었습니다.
사람이 본문 전체를 훑어 만든 `toc_docs/D0XX_toc.json`을 그대로 본문 JSON의 `toc` 필드에 덮어씁니다.

### 효과
검색 시스템이 문서의 실제 목차 체계(Ⅰ/1/2... 등 계층)를 갖게 되어, "이 문서의 목차가 무엇인가"를
물어보는 질의에도 정확히 답할 수 있게 됩니다. 이후 Phase 2(섹션 매칭)의 기준점이 됩니다.

### 안전장치
파일명(`D001_toc.json`)만 믿지 않고, JSON **내부에 적힌 `doc_id` 필드까지 이중 확인**합니다.
파일명과 내부 doc_id가 다르면(저장 실수) 병합을 중단하고 경고를 남깁니다.

In [ ]:
def merge_toc(doc_id: str, main_data: dict, toc_path: Path, toc_data: dict) -> dict:
    """toc_docs의 toc를 main_data['toc']에 덮어쓴다. doc_id 이중 확인 포함."""
    assert toc_data['doc_id'] == doc_id, (
        f'doc_id 불일치: 파일명 기준={doc_id}, 내부 필드={toc_data["doc_id"]} ({toc_path.name})'
    )
    main_data['toc'] = toc_data['toc']
    return main_data

merge_log = []

for doc_id in target_ids:
    _, main_data = docs_by_id[doc_id]
    toc_path, toc_data = toc_by_id[doc_id]
    merge_toc(doc_id, main_data, toc_path, toc_data)
    uncertain_cnt = sum(1 for t in toc_data['toc'] if t.get('is_uncertain'))
    merge_log.append({'doc_id': doc_id, 'toc_items': len(toc_data['toc']), 'is_uncertain_count': uncertain_cnt})

print(f'Phase 1 완료 : {len(target_ids)}건 toc 병합')
print('is_uncertain 많은 문서 상위 5건 (먼저 검수 추천):')
for row in sorted(merge_log, key=lambda r: -r['is_uncertain_count'])[:5]:
    print(' ', row)

제외된 두 문서 빼고 전체 병합 성공

---
##2.header_path ↔ toc 매칭

### 왜 하는가
본문 `sections[].header_path`는 글머리표(□, 가/나, 1)/2)) 기준으로 잘려 있어서, 방금 채운
진짜 목차(`toc`)와 표현이 다릅니다. 예를 들어 목차에는 `"추진목표"`라고만 적혀 있는데,
본문 섹션 제목은 `"□ 정보기술 활용 전략"`처럼 전혀 다른 문장일 수 있습니다.
이 둘을 텍스트 유사도로 비교해서 **가장 비슷한 목차 항목과 연결(`toc_ref`)**해줍니다.

### 효과
"목차 2.1. 추진목표에 해당하는 내용을 보여줘" 같은 질의가 들어왔을 때, 시스템이 `toc_ref`만
보고 정확한 섹션으로 바로 찾아갈 수 있습니다. 매칭이 안 되어 있으면 목차 구조와 본문 내용이
서로 단절된 채로 남습니다.

### 방식
- 외부 라이브러리(rapidfuzz) 설치 없이, 표준 라이브러리 `difflib.SequenceMatcher`로 유사도 계산
- 매칭 실패(임계값 미만)는 `toc_ref: null` + `toc_match_failed: true`로 표시 → 사람이 검수할 목록으로 별도 출력
- 기존 `header_path`는 삭제하지 않고 그대로 둡니다 (본문 표현이라는 정보 자체도 유의미하므로). `toc_ref`는 "목차상 위치"를 추가로 알려주는 보강 필드입니다.

##2.1.손으로 검수 후 재실행

before : 전체 섹션 6515건 중 매칭 성공 2909건
매칭 실패(검수 필요) : 3606건

after :

In [ ]:
from typing import Any # Import Any for more flexible type hinting

def similarity(a: Any, b: Any) -> float:
    # Ensure both a and b are strings. If they are None, convert to empty string.
    a_safe = str(a) if a is not None else ''
    b_safe = str(b) if b is not None else ''
    return SequenceMatcher(None, a_safe, b_safe).ratio()

def match_section_to_toc(header_text: str, toc_items: list, threshold: float = 0.45):
    """header_path 마지막 텍스트와 가장 유사한 toc 항목을 찾는다."""
    best, best_score = None, 0.0
    for item in toc_items:
        # Ensure item['title'] is a string, even if item['title'] is None or missing
        toc_title = item.get('title', '') if isinstance(item, dict) else ''

        score = similarity(header_text, toc_title)
        if score > best_score:
            best, best_score = item, score
    if best is not None and best_score >= threshold:
        return best, best_score
    return None, best_score

def attach_toc_ref(main_data: dict, threshold: float = 0.45) -> list:
    """모든 섹션에 toc_ref를 부여하고, 매칭 실패 섹션 목록을 반환한다."""
    failed = []
    toc_items = main_data.get('toc', [])
    for section in main_data['sections']:
        # Ensure header_text is always a string, even if the last element of header_path is None or header_path is empty/missing
        header_text = str(section['header_path'][-1]) if section.get('header_path') else ''

        matched, score = match_section_to_toc(header_text, toc_items, threshold)
        if matched:
            section['toc_ref'] = {
                'order': matched['order'],
                'number': matched['number'],
                'title': matched['title'],
                'match_score': round(score, 3),
            }
            section['toc_match_failed'] = False
        else:
            section['toc_ref'] = None
            section['toc_match_failed'] = True
            failed.append({
                'doc_id': main_data['doc_id'],
                'section_id': section['section_id'],
                'header_text': header_text,
                'best_score': round(score, 3),
            })
    return failed

all_match_failures = []
for doc_id in target_ids:
    _, main_data = docs_by_id[doc_id]
    failed = attach_toc_ref(main_data)
    all_match_failures.extend(failed)

matched_total = sum(
    1 for doc_id in target_ids
    for s in docs_by_id[doc_id][1]['sections']
    if not s['toc_match_failed']
)
section_total = sum(len(docs_by_id[doc_id][1]['sections']) for doc_id in target_ids)

print(f'Phase 2 완료 : 전체 섹션 {section_total}건 중 매칭 성공 {matched_total}건')
print(f'매칭 실패(검수 필요) : {len(all_match_failures)}건')
for row in all_match_failures[:10]:
    print(' ', row)

---
##3.표 병합 셀 중복 정리

### 왜 하는가
HWP/PDF에서 병합된 셀이 텍스트로 추출되면, 합쳐졌던 칸 수만큼 같은 텍스트가 옆으로
반복되어 들어옵니다 (예: `"목차 목차 목차 목차 목차 목차 목차"`). 이 상태로 청크에 들어가면
의미 없는 토큰만 늘어나고 임베딩 검색 품질도 떨어집니다.

### 효과
표 한 줄을 왼쪽부터 훑으면서 **바로 앞 칸과 같은 값이면 제거**하는 단순한 규칙만으로
병합 셀 중복을 안전하게 정리할 수 있습니다. 정리 후 1행 1열로 줄어든 표는 장식용 표로
재분류(`is_decorative`)합니다.



In [ ]:
def dedup_merged_cells(raw_grid: list) -> list:
    """같은 행 안에서 바로 앞 칸과 동일한 값이 연속되면 병합 셀 흔적으로 보고 제거"""
    cleaned = []
    for row in raw_grid:
        new_row = []
        prev = object()  # 첫 칸은 항상 통과시키기 위한 더미 값
        for cell in row:
            if cell == prev:
                continue
            new_row.append(cell)
            prev = cell
        cleaned.append(new_row)
    return cleaned

def grid_to_markdown(grid: list) -> str:
    """정리된 grid를 마크다운 표로 재직렬화"""
    if not grid:
        return ''
    rows = ['| ' + ' | '.join(str(c) for c in row) + ' |' for row in grid]
    header_sep = '| ' + ' | '.join(['---'] * len(grid[0])) + ' |'
    return '\n'.join([rows[0], header_sep] + rows[1:]) if len(rows) > 1 else rows[0] + '\n' + header_sep

def clean_tables(main_data: dict) -> int:
    """문서 내 모든 표 블록을 정리하고, 변경된 블록 수를 반환"""
    changed = 0
    for section in main_data['sections']:
        for block in section['blocks']:
            if block.get('type') != 'table':
                continue
            before = block.get('raw_grid', [])
            after = dedup_merged_cells(before)
            if after != before:
                block['raw_grid'] = after
                block['content'] = grid_to_markdown(after)
                changed += 1
            # 정리 후 1행 1열로 줄어들면 장식용 표로 재태깅
            flat = [c for row in block['raw_grid'] for c in row]
            if len(block['raw_grid']) <= 1 and len(flat) <= 1 and not block.get('is_decorative'):
                block['is_decorative'] = True
                block['decorative_reason'] = 'dedup_collapsed_to_single_cell'
    return changed

table_clean_log = []
for doc_id in target_ids:
    _, main_data = docs_by_id[doc_id]
    changed = clean_tables(main_data)
    if changed:
        table_clean_log.append({'doc_id': doc_id, 'tables_cleaned': changed})

print(f'Phase 3 완료 : 표가 정리된 문서 {len(table_clean_log)}건')
for row in table_clean_log[:10]:
    print(' ', row)

---
##4.대형 섹션 분할 표시

### 왜 하는가
한 섹션에 블록이 100개 넘게 몰려 있으면(예: D001의 S26 = 164블록), 청킹 담당자가 이 섹션을
그대로 청크 1개로 쓸 수 없습니다. 하지만 "어디서 끊을지"는 토큰 길이를 보며 판단해야 하는
청킹 영역의 일이므로, 여기서는 **분할이 필요하다는 신호만 남깁니다.**

### 효과
청킹 담당자가 `needs_subsplit: true`인 섹션만 골라서 우선 검토할 수 있어, 100개 문서
전체를 다시 훑지 않고도 큰 섹션을 빠르게 식별할 수 있습니다.

### Phase 2와의 관계
섹션 경계가 Phase 2에서 `toc_ref`로 재정렬된 뒤의 상태를 기준으로 판단해야, 실제로 목차상
온전한 하나의 챕터인데 블록만 많은 것인지 / 여러 챕터가 잘못 묶여 있는 것인지 구분할 수 있습니다.

In [ ]:
BLOCK_THRESHOLD = 30  # 기준값 — 분포 확인 후 조정 가능

def flag_large_sections(main_data: dict, threshold: int = BLOCK_THRESHOLD) -> list:
    flagged = []
    for section in main_data['sections']:
        block_count = len(section['blocks'])
        section['block_count'] = block_count
        section['needs_subsplit'] = block_count > threshold
        if section['needs_subsplit']:
            flagged.append({
                'doc_id': main_data['doc_id'],
                'section_id': section['section_id'],
                'block_count': block_count,
                'toc_ref': section.get('toc_ref'),
            })
    return flagged

large_section_log = []
for doc_id in target_ids:
    _, main_data = docs_by_id[doc_id]
    large_section_log.extend(flag_large_sections(main_data))

print(f'Phase 4 완료 : 분할 필요 섹션(block_count > {BLOCK_THRESHOLD}) {len(large_section_log)}건')
for row in sorted(large_section_log, key=lambda r: -r['block_count'])[:10]:
    print(' ', row)

---
##5.qa 재계산 & final_data 저장

### 왜 하는가
Phase 1~4를 거치며 toc, 섹션 매칭, 표, 분할 표시가 모두 바뀌었으므로, 처음 만들었던 `qa`
통계는 더 이상 현재 상태를 반영하지 못합니다. 모든 패치가 끝난 시점에 **한 번에 다시 계산**해야
의미가 있습니다.

### 효과
- `toc_header_match_rate`가 이번 작업의 핵심 성공 지표입니다. 1.0에 가까울수록 목차-본문 연결이
  잘 된 것이고, 낮게 나오는 문서는 Phase 2의 임계값(threshold)이나 toc 품질을 다시 점검해야 합니다.
- 100개 문서 전체의 품질을 사람이 일일이 다시 열어보지 않고, 이 표 하나로 빠르게 점검할 수 있습니다.

### 저장 위치
`Preprocessed dataset/final_data/D0XX.json` — `docs` 폴더의 원본은 건드리지 않고
별도 폴더에 최종본을 새로 저장합니다 (원본 보존 + 재현 가능성 확보).

In [ ]:
def recompute_qa(main_data: dict) -> dict:
    sections = main_data['sections']
    total_blocks  = sum(len(s['blocks']) for s in sections)
    table_blocks  = sum(1 for s in sections for b in s['blocks'] if b['type'] == 'table')
    text_blocks   = total_blocks - table_blocks
    decorative    = sum(1 for s in sections for b in s['blocks'] if b.get('is_decorative'))
    matched       = sum(1 for s in sections if not s.get('toc_match_failed'))
    needs_split   = sum(1 for s in sections if s.get('needs_subsplit'))

    qa = main_data.get('qa', {})
    qa.update({
        'total_sections': len(sections),
        'total_blocks': total_blocks,
        'text_blocks': text_blocks,
        'table_blocks': table_blocks,
        'decorative_table_count': decorative,
        'decorative_table_ratio': round(decorative / table_blocks, 3) if table_blocks else 0.0,
        'toc_header_match_rate': round(matched / len(sections), 3) if sections else 0.0,
        'needs_subsplit_count': needs_split,
    })
    main_data['qa'] = qa
    return qa

qa_summary = []
for doc_id in target_ids:
    path, main_data = docs_by_id[doc_id]
    qa = recompute_qa(main_data)

    out_path = FINAL_DIR / f'{doc_id}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(main_data, f, ensure_ascii=False, indent=2)

    qa_summary.append({'doc_id': doc_id, **qa})

print(f'Phase 5 완료 : {len(qa_summary)}건 final_data 저장 → {FINAL_DIR}')

In [ ]:
import pandas as pd

qa_df = pd.DataFrame(qa_summary)
print('toc_header_match_rate 분포:')
print(qa_df['toc_header_match_rate'].describe())

print('\n매칭률 낮은 문서 (재검수 우선순위) Top 10:')
print(qa_df.sort_values('toc_header_match_rate').head(10)[
    ['doc_id', 'toc_header_match_rate', 'needs_subsplit_count']
].to_string(index=False))

qa_summary_path = OUTPUT_DIR / 'phase2_qa_summary.csv'
qa_df.to_csv(qa_summary_path, index=False, encoding='utf-8-sig')
print(f'\nQA 요약 저장 : {qa_summary_path}')

---
## 최종 검증

특정 문서를 골라 병합·정리가 의도대로 적용됐는지 직접 확인합니다. `CHECK_DOC_ID`를 바꿔가며 점검하세요.

In [ ]:
CHECK_DOC_ID = 'D001'

with open(FINAL_DIR / f'{CHECK_DOC_ID}.json', encoding='utf-8') as f:
    check = json.load(f)

print('toc 항목 수 :', len(check['toc']))
print('toc 상위 5개 :')
for t in check['toc'][:5]:
    print(' ', t)

print('\n섹션별 toc_ref 매칭 상태 (앞 5개):')
for s in check['sections'][:5]:
    print(' ', s['section_id'], '→', s.get('toc_ref'), '| needs_subsplit:', s.get('needs_subsplit'))

print('\nqa 요약 :', check['qa'])

---
## 요약

| 항목 | 내용 |
|---|---|
| 처리 대상 | 100건 중 `D026`, `D043` 제외, `docs`/`toc_docs` 양쪽에 존재하는 문서 |
| 입력 ① | `Preprocessed dataset/docs/*.json` (1차 전처리 결과) |
| 입력 ② | `Preprocessed dataset/toc_docs/*_toc.json` (사람이 만든 목차, doc_id로 연결) |
| 출력 | `Preprocessed dataset/final_data/D0XX.json` (병합 + 정리 완료) |
| 부가 출력 | `Preprocessed dataset/phase2_qa_summary.csv` (문서별 toc_header_match_rate 등 품질 지표) |

다음 작업은 `phase2_qa_summary.csv`에서 `toc_header_match_rate`가 낮은 문서를 우선 검수하고,
필요하면 Phase 2의 `threshold` 값을 조정해 재실행하는 것을 추천합니다.